# 74 — Campaign C staged M3（child M2 完走後）

前提: notebook 73 で child M2 が `M2_COMPLETE`、`audit/m2_accepted_parent.json` が書換済み。

- **CUDA GPU 必須**（`require_cuda=True`）
- 親は **今回の child M2**（j2_max=2）。デフォルト j2=1 親は使わない
- 成果は `CORE_REPRODUCED` まで。常に `NOT_CERTIFIED`
- RSVD / Triad / influence proxy は探索値。q&lt;1 の証明ではない


In [ ]:
from pathlib import Path
import os
import sys

BUNDLE_NAME = 'validated_4d_su2_rg_codex_bundle'
explicit = os.environ.get('VALIDATED_RG_PROJECT_ROOT')
candidates = ([Path(explicit)] if explicit else []) + [
    Path.cwd(), Path.cwd().parent, Path.cwd() / BUNDLE_NAME,
    Path('/notebooks') / BUNDLE_NAME, Path('/notebooks'),
]
PROJECT_ROOT = next((
    p.expanduser().resolve()
    for p in candidates
    if (p.expanduser() / 'src' / 'm3_orchestrator.py').is_file()
), None)
if PROJECT_ROOT is None:
    raise RuntimeError('Pull latest main; src/m3_orchestrator.py not found.')
PERSIST_ROOT = Path(
    os.environ.get('VALIDATED_RG_PERSIST_ROOT', '/storage/validated_4d_su2_rg')
).expanduser().resolve()
os.environ['VALIDATED_RG_PROJECT_ROOT'] = str(PROJECT_ROOT)
os.environ['VALIDATED_RG_PERSIST_ROOT'] = str(PERSIST_ROOT)
os.environ.setdefault(
    'VALIDATED_RG_PERSIST_ACK', 'I_CONFIRM_THIS_PATH_IS_PERSISTENT',
)
os.environ.setdefault('VALIDATED_RG_M7C_RUN_ID', 'M7-20260720T081500Z-c8d5f02b3c96')
os.environ.setdefault('VALIDATED_RG_STAGED_CANDIDATE', 'CAND-000005-ac85c5e9ba38')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import torch
print('PROJECT_ROOT', PROJECT_ROOT)
print('PERSIST_ROOT', PERSIST_ROOT)
print('torch', torch.__version__, 'CUDA', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('M3 child lineage requires CUDA GPU.')
print('GPU', torch.cuda.get_device_properties(0).name)


In [ ]:
import json
from pathlib import Path

from src.common import read_json

M7C_RUN_ID = os.environ.get('VALIDATED_RG_M7C_RUN_ID', 'M7-20260720T081500Z-c8d5f02b3c96')
DEFAULT_CAND = os.environ.get('VALIDATED_RG_STAGED_CANDIDATE', 'CAND-000005-ac85c5e9ba38')
explicit_pkg = os.environ.get('VALIDATED_RG_STAGED_PACKAGE')
PACKAGE_ROOT = (
    Path(explicit_pkg).expanduser().resolve()
    if explicit_pkg else
    (PERSIST_ROOT / 'searches' / M7C_RUN_ID / 'auto_execute' / DEFAULT_CAND).resolve()
)
required = [
    'scheme.json', 'resource_gate.json', 'child_run_ids.json',
    'm3_config_overrides.json',
]
missing = [n for n in required if not (PACKAGE_ROOT / n).is_file()]
if missing:
    raise FileNotFoundError(f'Package incomplete: {PACKAGE_ROOT} missing {missing}')

CHILD_IDS = read_json(PACKAGE_ROOT / 'child_run_ids.json')
M2_RUN_ID = CHILD_IDS['M2']
M3_RUN_ID = CHILD_IDS['M3']
os.environ['VALIDATED_RG_M2_RUN_ID'] = M2_RUN_ID
os.environ['VALIDATED_RG_M3_RUN_ID'] = M3_RUN_ID

m2_acceptance = PERSIST_ROOT / 'runs' / M2_RUN_ID / 'reports' / 'M2_acceptance.json'
if not m2_acceptance.is_file():
    raise FileNotFoundError(
        f'Child M2 not complete: missing {m2_acceptance}. Finish notebook 73 first.'
    )
print('PACKAGE_ROOT', PACKAGE_ROOT)
print('M2_RUN_ID', M2_RUN_ID)
print('M3_RUN_ID', M3_RUN_ID)


## M3Config（child M2 親 + package overrides）


In [ ]:
from dataclasses import asdict
from pathlib import Path

from src.cutoff_dims import cutoff_dimension_payload
from src.m3_config import M3Config
from src.m3_parent import verify_accepted_m2_parent

over = read_json(PACKAGE_ROOT / 'm3_config_overrides.json')
from src.m7_staged_lineage import write_child_m2_acceptance_audit

audit_path = PROJECT_ROOT / 'audit' / 'm2_accepted_parent.json'
audit = read_json(audit_path) if audit_path.is_file() else None
if not isinstance(audit, dict) or audit.get('accepted_run_id') != M2_RUN_ID:
    print(
        'Rewriting audit/m2_accepted_parent.json from child M2:',
        M2_RUN_ID,
        '(was', None if not isinstance(audit, dict) else audit.get('accepted_run_id'), ')',
    )
    audit = write_child_m2_acceptance_audit(
        PROJECT_ROOT,
        run_root=PERSIST_ROOT / 'runs' / M2_RUN_ID,
    )
if not isinstance(audit, dict):
    raise RuntimeError(f'Malformed audit: {audit_path}')
if audit.get('accepted_run_id') != M2_RUN_ID:
    raise RuntimeError(
        f"audit accepted_run_id={audit.get('accepted_run_id')!r} != package M2={M2_RUN_ID!r} "
        'after rewrite.'
    )

dims = cutoff_dimension_payload(int(over['j2_max']))
base = asdict(M3Config())
base.update({
    'parent_run_id': audit['accepted_run_id'],
    'parent_checkpoint': Path(audit['checkpoint_path']).name,
    'parent_checkpoint_path': audit['checkpoint_path'],
    'parent_report_path': audit['m2_report_path'],
    'parent_acceptance_path': audit['m2_acceptance_path'],
    'j2_max': int(over['j2_max']),
    'sector_count': int(over['sector_count']),
    'operator_dimension': int(over['operator_dimension']),
    'target_rank': int(over['target_rank']),
    'require_cuda': True,
})
if base['sector_count'] != dims['sector_count']:
    raise RuntimeError('m3_config_overrides.sector_count disagrees with cutoff_dims.')
if base['operator_dimension'] != dims['operator_dimension']:
    raise RuntimeError('m3_config_overrides.operator_dimension disagrees with cutoff_dims.')

M3_CONFIG = M3Config(**base)
M2_EVIDENCE = verify_accepted_m2_parent(PROJECT_ROOT, M3_CONFIG)
print(json.dumps({
    'parent_run': M3_CONFIG.parent_run_id,
    'parent_checkpoint': M3_CONFIG.parent_checkpoint,
    'j2_max': M3_CONFIG.j2_max,
    'sector_count': M3_CONFIG.sector_count,
    'operator_dimension': M3_CONFIG.operator_dimension,
    'target_rank': M3_CONFIG.target_rank,
    'projector_tensors': len(M2_EVIDENCE.projector_tensors),
    'certification_status': M3_CONFIG.certification_status,
}, indent=2, ensure_ascii=False))


## テスト報告

再開で時間が惜しいときは `SKIP_M3_TESTS=1`（既存 `test_report.json` を再利用）。


In [ ]:
import time
import pytest
from src.common import atomic_write_json, read_json

run_root = PERSIST_ROOT / 'runs' / M3_RUN_ID
saved = run_root / 'test_report.json'
skip = os.environ.get('SKIP_M3_TESTS', '').strip() in {'1', 'true', 'TRUE', 'yes'}
if skip and saved.is_file():
    M3_TEST_REPORT = read_json(saved)
    print('Reusing saved test_report.json')
else:
    started = time.monotonic()
    os.chdir(PROJECT_ROOT)
    cpu_rc = pytest.main([
        '-q', f'--rootdir={PROJECT_ROOT}', '-m', 'not gpu',
        # M3 gate: pin M0–M3 relevant tests (avoid unrelated M6 suite).
        'tests/test_m2_batching.py',
        'tests/test_m2_fusion.py',
        'tests/test_m2_restart.py',
        'tests/test_armillary_equivalence.py',
        'tests/test_su2_multiplicity.py',
        'tests/test_generator_action.py',
        'tests/test_matrix_free.py',
        'tests/test_m3_restart.py',
    ])
    if cpu_rc != 0:
        raise RuntimeError(f'M0–M3 CPU suite failed: {cpu_rc}')
    gpu_rc = pytest.main([
        '-q', f'--rootdir={PROJECT_ROOT}',
        'tests/test_matrix_free.py', 'tests/test_m3_restart.py', '-m', 'gpu',
    ])
    if gpu_rc != 0:
        raise RuntimeError(f'M3 required GPU suite failed: {gpu_rc}')
    M3_TEST_REPORT = {
        'accepted_m2_parent': 'PASS',
        'm0_m1_m2_regression_cpu_suite': 'PASS',
        'm3_required_cpu_suite': 'PASS',
        'm3_required_gpu_suite': 'PASS',
        'm3_fresh_process_resume': 'PASS',
        'm3_checkpoint_basis_restore': 'PASS',
        'm3_oom_recovery': 'PASS',
        'elapsed_s': time.monotonic() - started,
        'suite': 'staged_m3_notebook',
    }
    run_root.mkdir(parents=True, exist_ok=True)
    atomic_write_json(saved, M3_TEST_REPORT)
print(json.dumps(M3_TEST_REPORT, indent=2))


## M3 セッション実行（checkpoint まで）

未完了なら **同じセルを再実行**（同じ `M3_RUN_ID` で resume）。
6時間制限・15分 checkpoint は orchestrator 側のままです。


In [ ]:
from src.m3_orchestrator import create_or_resume_m3
from src.runtime import environment_info, validate_persistent_root

PERSIST = validate_persistent_root()
print(json.dumps(environment_info(), ensure_ascii=False, indent=2, default=str))

m3_orchestrator = create_or_resume_m3(
    PERSIST,
    M3_CONFIG,
    PROJECT_ROOT,
    run_id=M3_RUN_ID,
    test_report=M3_TEST_REPORT,
)
print('run_id', m3_orchestrator.state.run_id)
print('phase', m3_orchestrator.state.phase)
assert m3_orchestrator.state.certification_status == 'NOT_CERTIFIED'

M3_RESULT = m3_orchestrator.run_until_checkpoint()
assert M3_RESULT['certification_status'] == 'NOT_CERTIFIED'
print(json.dumps(M3_RESULT, indent=2, ensure_ascii=False, default=str))


## 進捗確認（何度でも可）


In [ ]:
from src.common import read_json

loaded = m3_orchestrator.checkpoints.load_latest(restore_rng=False)
if loaded is None:
    raise RuntimeError('No valid M3 checkpoint yet.')
inspection = {
    'run_id': loaded.state.run_id,
    'phase': loaded.state.phase,
    'checkpoint': str(loaded.path),
    'checkpoint_index': loaded.state.checkpoint_index,
    'done': sum(item.status == 'done' for item in loaded.queue.items.values()),
    'pending': sum(item.status == 'pending' for item in loaded.queue.items.values()),
    'failed': sum(item.status == 'failed' for item in loaded.queue.items.values()),
    'candidate_tensors': sorted(loaded.tensors),
    'certification_status': loaded.state.certification_status,
}
print(json.dumps(inspection, indent=2, ensure_ascii=False, default=str))

report_path = m3_orchestrator.run_root / 'reports' / 'M3_report.json'
if report_path.is_file():
    report = read_json(report_path)
    rsvd = (report.get('results') or {}).get('M3_RSVD', {}).get('result', {})
    print(json.dumps({
        '判定': 'M3 report present (exploratory; NOT_CERTIFIED).',
        'phase': report.get('phase'),
        'acceptance_gates': report.get('acceptance_gates'),
        'singular_values_head': (rsvd.get('singular_values') or [])[:8],
        'relative_residual': rsvd.get('relative_residual_frobenius'),
        'influence_proxy': rsvd.get('influence_proxy'),
        'heuristic_results': report.get('heuristic_results'),
        'certification_status': report.get('certification_status'),
    }, indent=2, ensure_ascii=False, default=str))
else:
    print('M3 incomplete — re-run the session cell with the same M3_RUN_ID.')
